In [1]:
# =============================================================================
# STEP 9b (nb50) - WHEN CAN REWEIGHTING REPAIR CONFORMAL COVERAGE?
#
# nb49 established that mixture-matched calibration fails on NSL-KDD R2L, and
# established WHY: label-shift estimation requires P(X | Y = c) to be invariant,
# and the class-conditional shift statistic of Section 5.10 measures precisely
# its violation. For R2L that statistic is 0.9905, so the assumption fails about
# as completely as it can, on exactly the class where coverage fails worst.
#
# One class is an anecdote. This notebook tests whether that relationship holds
# across all four environments, turning a failed remedy into a stated boundary
# condition with a decision rule attached:
#
#     S_cov,c is a FEASIBILITY TEST for reweighting-based repair.
#     Where it is near the permutation null, the mixture is recoverable and
#     reweighting is defensible. Where it approaches one, no reweighting scheme
#     that estimates target composition from a source-trained model can work,
#     and this is checkable BEFORE attempting the repair.
#
# HYPOTHESIS, stated before the result: mixture-estimation error rises with
# S_cov,c. The class-level prior is the textbook label-shift setting, so it
# applies to all four environments and yields nineteen classes spanning S_cov,c
# from 0.492 to 1.000.
#
# THIS CAN FAIL. If estimation error is unrelated to S_cov,c, the nb49 result is
# a one-dataset curiosity and there is no boundary condition to state.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
from scipy.optimize import nnls
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
import os, sys, json, shutil, glob, subprocess, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
print('ready')


Mounted at /content/drive
ready


In [2]:
# =============================================================================
# Cell 2 - estimator and per-class error, identical machinery to nb49 so the
# results are directly comparable.
# =============================================================================
def confusion_cv(X, y, K, seed=0, folds=5):
    cnt=np.bincount(y, minlength=K); usable=cnt[cnt>0]
    k=int(min(folds, usable.min())) if len(usable) else 0
    if k<2: return None
    C=np.zeros((K,K)); tot=np.zeros(K)
    for tr,te in StratifiedKFold(k, shuffle=True, random_state=seed).split(X,y):
        m=RandomForestClassifier(n_estimators=250, class_weight='balanced',
                                 n_jobs=-1, random_state=seed).fit(X[tr],y[tr])
        pr=m.predict(X[te])
        for t,p in zip(y[te],pr): C[p,t]+=1
        for t in y[te]: tot[t]+=1
    for j in range(K):
        if tot[j]>0: C[:,j]/=tot[j]
    return C

def estimate(C, p_hat):
    q,_=nnls(C, p_hat); s=q.sum()
    return q/s if s>1e-12 else np.full(len(q), 1.0/len(q))

def tv(a,b): return 0.5*float(np.abs(np.asarray(a)-np.asarray(b)).sum())

def per_class_error(q_hat, q_true, classes):
    """Error attributable to each class, so it can be paired with that class's S_cov,c."""
    return {c: abs(float(q_hat[i]-q_true[i])) for i,c in enumerate(classes)}

def run_env(name, Xs, ys, Xt, yt, classes, seed=0):
    K=len(classes)
    C=confusion_cv(Xs, ys, K, seed=seed)
    if C is None: return None
    mdl=RandomForestClassifier(n_estimators=250, class_weight='balanced',
                               n_jobs=-1, random_state=seed).fit(Xs, ys)
    pr=mdl.predict(Xt)
    p_hat=np.array([(pr==i).mean() for i in range(K)])
    q_hat=estimate(C, p_hat)
    q_true=np.array([(yt==i).mean() for i in range(K)])
    p_src =np.array([(ys==i).mean() for i in range(K)])
    return {'name':name,'classes':classes,'C':C,'q_hat':q_hat,'q_true':q_true,
            'p_src':p_src,'p_hat':p_hat,
            'tv_est':tv(q_hat,q_true),'tv_raw':tv(p_hat,q_true),'tv_none':tv(p_src,q_true),
            'per_class':per_class_error(q_hat,q_true,classes),
            'cond':float(np.linalg.cond(C))}
print('estimator ready')


estimator ready


In [3]:
# =============================================================================
# Cell 3 - build source/target feature matrices for all four environments.
# =============================================================================
DROP={'label','subtype','partition','is_unseen'}
ENVS={}

# ---- NSL-KDD ----
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
F=[c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]
s=tr[tr.partition=='source_cal_pool']
ENVS['nslkdd']=(s[F].to_numpy(float), s['label'].map(c2i).to_numpy(),
                te[F].to_numpy(float), te['label'].map(c2i).to_numpy(), CL)

# ---- UGR'16 ----
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for d in (us,ut): d['label']=d['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
FU=[c for c in us.columns if c not in DROP and pd.api.types.is_numeric_dtype(us[c])]
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,x in df.groupby(col,sort=True):
        idx=x.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
usp=us[us.partition=='source_cal_pool']
ENVS['ugr16']=(usp[FU].to_numpy(float), usp['label'].map(U2I).to_numpy(),
               ut[FU].to_numpy(float), ut['label'].map(U2I).to_numpy(), UCL)

# ---- CIC-IoT-2023 ----
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
FI=json.loads((RD/'ciciot2023_prepared_fingerprint.json').read_text())['features']
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
i2i={c:i for i,c in enumerate(ICL)}
isp=iot[iot.partition=='source_cal_pool']; itg=iot[iot.partition=='target_pool']
ENVS['ciciot2023']=(isp[FI].to_numpy(float), isp['family'].map(i2i).to_numpy(),
                    itg[FI].to_numpy(float), itg['family'].map(i2i).to_numpy(), ICL)

# ---- CIC-IDS2017 (first realisation; per-realisation splits) ----
import features_cic as fc
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True)
wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
FC=fc.feature_cols(wed)
Xc=np.column_stack([pd.to_numeric(wed[c],errors='coerce').to_numpy(float) for c in FC])
Xc=np.nan_to_num(Xc, nan=0.0, posinf=0.0, neginf=0.0)
lab=(wed['label']=='DoS').astype(int).to_numpy()
spx=np.load(config.PROC_DIR/'cic_R1_holdout_Slowhttptest_srcpool_idx.npy')
tgx=np.load(config.PROC_DIR/'cic_R1_holdout_Slowhttptest_target_idx.npy')
ENVS['cicids2017']=(Xc[spx], lab[spx], Xc[tgx], lab[tgx], ['Benign','DoS'])

for k,(Xs,ys,Xt,yt,cl) in ENVS.items():
    print(f"  {k:12s} source {len(ys):>7,} target {len(yt):>7,} classes {len(cl)}")


  nslkdd       source  18,894 target  22,544 classes 5
  ugr16        source  60,000 target 400,000 classes 5
  ciciot2023   source 158,079 target 456,174 classes 8
  cicids2017   source  48,459 target 154,794 classes 2


In [4]:
# =============================================================================
# Cell 4 - estimate every environment, then pair per-class error with S_cov,c.
# =============================================================================
res={}; t0=time.time()
for k,(Xs,ys,Xt,yt,cl) in ENVS.items():
    # cap for tractability; stratified so rare classes survive
    rg=np.random.default_rng(0)
    if len(ys)>40000:
        idx=rg.choice(len(ys),40000,replace=False); Xs,ys=Xs[idx],ys[idx]
    if len(yt)>60000:
        idx=rg.choice(len(yt),60000,replace=False); Xt,yt=Xt[idx],yt[idx]
    r=run_env(k,Xs,ys,Xt,yt,cl,seed=0)
    if r is None: print(f'  {k}: not estimable'); continue
    res[k]=r
    print(f"  {k:12s} TV(est)={r['tv_est']:.4f}  TV(raw)={r['tv_raw']:.4f}  "
          f"TV(no shift)={r['tv_none']:.4f}  cond(C)={r['cond']:.1f}  [{time.time()-t0:.0f}s]")

rows=[]
for k,r in res.items():
    for i,c in enumerate(r['classes']):
        rows.append({'dataset':k,'class':c,'est_share':r['q_hat'][i],'true_share':r['q_true'][i],
                     'source_share':r['p_src'][i],'abs_error':abs(r['q_hat'][i]-r['q_true'][i])})
E=pd.DataFrame(rows)
cc=pd.read_csv(RD/'class_conditional_scov_vs_coverage.csv')[['dataset','class','S_cov_class','coverage','undercoverage']]
M=E.merge(cc, on=['dataset','class'], how='inner')
print(f"\npaired classes: {len(M)}")
print(M[['dataset','class','S_cov_class','source_share','true_share','est_share','abs_error']]
      .round(4).sort_values('S_cov_class').to_string(index=False))


  nslkdd       TV(est)=0.2262  TV(raw)=0.2273  TV(no shift)=0.1376  cond(C)=2.2  [18s]
  ugr16        TV(est)=0.0478  TV(raw)=0.0469  TV(no shift)=0.0024  cond(C)=1.0  [67s]
  ciciot2023   TV(est)=0.0086  TV(raw)=0.0314  TV(no shift)=0.0135  cond(C)=5.0  [190s]
  cicids2017   TV(est)=0.0024  TV(raw)=0.0021  TV(no shift)=0.5137  cond(C)=1.0  [320s]

paired classes: 19
   dataset       class  S_cov_class  source_share  true_share  est_share  abs_error
ciciot2023      Benign       0.4919        0.0416      0.0396     0.0400     0.0004
ciciot2023        DDoS       0.4949        0.4334      0.4276     0.4284     0.0008
ciciot2023       Mirai       0.4958        0.1202      0.1170     0.1169     0.0001
ciciot2023    Spoofing       0.4984        0.0799      0.0794     0.0771     0.0023
cicids2017      Benign       0.4992        0.4751      0.9888     0.9912     0.0024
ciciot2023         DoS       0.4994        0.1567      0.1596     0.1595     0.0000
ciciot2023       Recon       0.5025       

In [5]:
# =============================================================================
# Cell 5 - THE TEST. Does class-conditional shift predict when reweighting can work?
# =============================================================================
r_all,p_all=stats.spearmanr(M.S_cov_class, M.abs_error)
print(f"S_cov,c vs per-class mixture-estimation error: rho {r_all:+.3f}  p {p_all:.4f}  n {len(M)}")

# dataset-level: overall estimation error against mean class-conditional shift
D=M.groupby('dataset').agg(mean_scov=('S_cov_class','mean'), max_scov=('S_cov_class','max')).reset_index()
D['tv_est']=[res[k]['tv_est'] for k in D.dataset]
D['tv_none']=[res[k]['tv_none'] for k in D.dataset]
D['helps']=D.tv_est < D.tv_none
print("\nDATASET LEVEL")
print(D.round(4).to_string(index=False))
r_d,p_d=stats.spearmanr(D.max_scov, D.tv_est) if len(D)>2 else (np.nan,np.nan)
print(f"  max S_cov,c vs total-variation error: rho {r_d:+.3f} (n={len(D)})")

print("\nTHE DECISION RULE")
lo=M[M.S_cov_class<0.70]; hi=M[M.S_cov_class>=0.90]
print(f"  classes with S_cov,c < 0.70 (n={len(lo)}): mean error {lo.abs_error.mean():.4f}, "
      f"max {lo.abs_error.max():.4f}")
print(f"  classes with S_cov,c >= 0.90 (n={len(hi)}): mean error {hi.abs_error.mean():.4f}, "
      f"max {hi.abs_error.max():.4f}")
if len(lo)>2 and len(hi)>2:
    u,pu=stats.mannwhitneyu(hi.abs_error, lo.abs_error, alternative='greater')
    print(f"  one-sided Mann-Whitney, high vs low: p {pu:.4f}")

print("\nVERDICT")
if r_all>0.3 and p_all<0.10:
    print("  Class-conditional shift PREDICTS whether the target mixture can be recovered.")
    print("  S_cov,c is therefore a feasibility test for reweighting-based repair, checkable")
    print("  before any repair is attempted. Where it sits near the permutation null the")
    print("  mixture is recoverable; where it approaches one, no scheme that estimates target")
    print("  composition from a source-trained model can work, because the assumption such")
    print("  schemes rest on is exactly what S_cov,c measures the violation of.")
elif r_all>0.3:
    print("  The relationship points the predicted way but is not resolved at this sample")
    print("  size. Report the direction and the estimate, not a rule.")
else:
    print("  Class-conditional shift does NOT predict estimation error. The nb49 failure is")
    print("  then specific to NSL-KDD R2L and there is no boundary condition to state.")
    print("  Report this against the hypothesis.")


S_cov,c vs per-class mixture-estimation error: rho +0.700  p 0.0008  n 19

DATASET LEVEL
   dataset  mean_scov  max_scov  tv_est  tv_none  helps
cicids2017     0.7494    0.9997  0.0024   0.5137   True
ciciot2023     0.5183    0.6536  0.0086   0.0135   True
    nslkdd     0.9295    0.9905  0.2262   0.1376  False
     ugr16     0.7552    0.9997  0.0478   0.0024  False
  max S_cov,c vs total-variation error: rho +0.000 (n=4)

THE DECISION RULE
  classes with S_cov,c < 0.70 (n=12): mean error 0.0019, max 0.0069
  classes with S_cov,c >= 0.90 (n=6): mean error 0.0531, max 0.1280
  one-sided Mann-Whitney, high vs low: p 0.0004

VERDICT
  Class-conditional shift PREDICTS whether the target mixture can be recovered.
  S_cov,c is therefore a feasibility test for reweighting-based repair, checkable
  before any repair is attempted. Where it sits near the permutation null the
  mixture is recoverable; where it approaches one, no scheme that estimates target
  composition from a source-trained mod

In [6]:
# =============================================================================
# Cell 6 - save and commit.
# =============================================================================
M.to_csv(RD/'mixture_feasibility_by_class.csv', index=False)
D.to_csv(RD/'mixture_feasibility_by_dataset.csv', index=False)
(RD/'mixture_feasibility_verdict.json').write_text(json.dumps({
 'hypothesis':'mixture-estimation error rises with class-conditional covariate shift, '
              'because label-shift estimation assumes P(X|Y=c) invariant and S_cov,c '
              'measures the violation of exactly that assumption',
 'n_classes':int(len(M)),
 'spearman_scov_vs_error':{'rho':float(r_all),'p':float(p_all)},
 'dataset_level':D.round(5).to_dict('records'),
 'low_shift_mean_error':float(lo.abs_error.mean()) if len(lo) else None,
 'high_shift_mean_error':float(hi.abs_error.mean()) if len(hi) else None,
 'interpretation':'if the relationship holds, S_cov,c is a pre-hoc feasibility test for '
                  'reweighting-based conformal repair',
 'nb49_anchor':'NSL-KDD R2L: S_cov,c 0.9905, mixture TV error 0.9411, and even a perfect '
               'estimate would give ESS 10.9 against a feasibility floor of 19'},
 indent=2, default=str))
print('saved feasibility tables and verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','nb50: class-conditional shift as a feasibility test for reweighting-based conformal repair')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved feasibility tables and verdict
[main f0ac44e] nb50: class-conditional shift as a feasibility test for reweighting-based conformal repair
 5 files changed, 73 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/50_mixture_feasibility.ipynb
 create mode 100644 reports/mixture_feasibility_by_class.csv
 create mode 100644 reports/mixture_feasibility_by_dataset.csv
 create mode 100644 reports/mixture_feasibility_verdict.json
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   c22bcf7..f0ac44e  main -> main
f0ac44e nb50: class-conditional shift as a feasibility test for reweighting-based conformal repair
c22bcf7 nb49: label-free subtype mixture estimation; gate for mixture-matched conformal calibration
34f0363 step 7c: threshold-referenced monitor; measures crossing of the calibrated quantile rather than movement anywhere

